# **Section 1: Database Connection**

In [ ]:
import pandas as pd
import sqlite3
conn = sqlite3.connect('healthcare_analytics.db')
print("Database Connected Successfully!")

Database Connected Successfully!


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('healthcare_analytics.db')

# Load data from CSV files into DataFrames
patient_df = pd.read_csv('patient_journey.csv')
provider_df = pd.read_csv('provider_master.csv')
comm_df = pd.read_csv('communication_logs.csv')
treatment_df = pd.read_csv('treatment_outcomes.csv')
country_df = pd.read_csv('country_reference.csv')

patient_df.to_sql(
    'patient_journey',
    conn,
    if_exists='replace',
    index=False
)
provider_df.to_sql(
    'provider_master',
    conn,
    if_exists='replace',
    index=False
)
comm_df.to_sql(
    'communication_logs',
    conn,
    if_exists='replace',
    index=False
)
treatment_df.to_sql(
    'treatment_outcomes',
    conn,
    if_exists='replace',
    index=False
)
country_df.to_sql(
    'country_reference',
    conn,
    if_exists='replace',
    index=False
)
print("Database tables created successfully!")

Database tables created successfully!


# **Section 2: View Available Tables**
2.1 List All Tables

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""
tables = pd.read_sql(query, conn)
display(tables)

,name
0,patient_journey
1,provider_master
2,communication_logs
3,treatment_outcomes
4,country_reference


2.2 Row Count for Each Table

In [8]:
tables = [
    'patient_journey',
    'provider_master',
    'communication_logs',
    'treatment_outcomes',
    'country_reference'
]
for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS total_records FROM {table}",
        conn
    )
    print(table)
    display(count)

patient_journey


,total_records
0,1500


provider_master


,total_records
0,30


communication_logs


,total_records
0,6025


treatment_outcomes


,total_records
0,554


country_reference


,total_records
0,12


# **Section 3: Funnel KPI Analysis**
3.1 Total Inquiries

In [9]:
query = """
SELECT COUNT(*) AS total_inquiries
FROM patient_journey;
"""
pd.read_sql(query, conn)

,total_inquiries
0,1500


3.2 Consultation Conversion

In [10]:
query = """
SELECT
COUNT(*) AS total_patients,
SUM(consultation_booked) AS consultations,
ROUND(
100.0 * SUM(consultation_booked)
/ COUNT(*),
2
) AS consultation_rate
FROM patient_journey;
"""
pd.read_sql(query, conn)

,total_patients,consultations,consultation_rate
0,1500,984,65.6


3.3 Quote Conversion

In [11]:
query = """
SELECT
SUM(quote_shared) AS total_quotes,
ROUND(
100.0 * SUM(quote_shared)
/ COUNT(*),
2
) AS quote_rate
FROM patient_journey;
"""
pd.read_sql(query, conn)

,total_quotes,quote_rate
0,561,37.4


3.4 Treatment Completion Rate

In [12]:
query = """
SELECT
SUM(treatment_completed) AS completed,
ROUND(
100.0 * SUM(treatment_completed)
/ COUNT(*),
2
) AS completion_rate
FROM patient_journey;
"""
pd.read_sql(query, conn)

,completed,completion_rate
0,554,36.93


3.5 Funnel Summary

In [13]:
query = """
SELECT
COUNT(*) AS inquiries,
SUM(consultation_booked) AS consultations,
SUM(quote_shared) AS quotes,
SUM(treatment_completed) AS treatments
FROM patient_journey;
"""
pd.read_sql(query, conn)

,inquiries,consultations,quotes,treatments
0,1500,984,561,554


# **Section 4: Provider Scorecard**
4.1 Provider Conversion Performance

In [14]:
query = """
SELECT
provider_id,
COUNT(*) AS patients,
SUM(treatment_completed) AS completed_treatments,
ROUND(
100.0 * SUM(treatment_completed)
/
COUNT(*),
2
) AS conversion_rate
FROM patient_journey
GROUP BY provider_id
ORDER BY conversion_rate DESC;
"""
pd.read_sql(query, conn)

,provider_id,patients,completed_treatments,conversion_rate
0,PVD-020,55,29,52.73
1,PVD-007,50,26,52.00
2,PVD-008,55,28,50.91
3,PVD-001,55,26,47.27
4,PVD-011,49,23,46.94
5,PVD-017,43,20,46.51
6,PVD-015,56,26,46.43
7,PVD-003,59,26,44.07
8,PVD-022,47,20,42.55
9,PVD-021,51,21,41.18


4.2 Average Response Time

In [15]:
query = """
SELECT
provider_id,
ROUND(
AVG(response_time_hours),
2
) AS avg_response_time
FROM patient_journey
GROUP BY provider_id
ORDER BY avg_response_time;
"""
pd.read_sql(query, conn)

,provider_id,avg_response_time
0,PVD-020,5.85
1,PVD-008,10.28
2,PVD-021,11.04
3,PVD-024,12.18
4,PVD-015,12.51
5,PVD-022,13.34
6,PVD-011,13.56
7,PVD-003,14.89
8,PVD-027,14.97
9,PVD-017,15.23


4.3 Top Performing Providers

In [16]:
query = """
SELECT
provider_id,
COUNT(*) AS patients,
SUM(treatment_completed) AS completed,
ROUND(
100.0 * SUM(treatment_completed)
/ COUNT(*),
2
) AS conversion_rate
FROM patient_journey
GROUP BY provider_id
HAVING COUNT(*) >= 5
ORDER BY conversion_rate DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,provider_id,patients,completed,conversion_rate
0,PVD-020,55,29,52.73
1,PVD-007,50,26,52.00
2,PVD-008,55,28,50.91
3,PVD-001,55,26,47.27
4,PVD-011,49,23,46.94
5,PVD-017,43,20,46.51
6,PVD-015,56,26,46.43
7,PVD-003,59,26,44.07
8,PVD-022,47,20,42.55
9,PVD-021,51,21,41.18


# **Section 5: Revenue Analytics**
5.1 Total Revenue

In [17]:
query = """
SELECT
ROUND(
SUM(actual_revenue_inr),
2
) AS total_revenue
FROM treatment_outcomes;
"""
pd.read_sql(query, conn)

,total_revenue
0,248325245.0


5.2 Revenue by Provider

In [18]:
query = """
SELECT
provider_id,
ROUND(
SUM(actual_revenue_inr),
2
) AS revenue
FROM treatment_outcomes
GROUP BY provider_id
ORDER BY revenue DESC;
"""
pd.read_sql(query, conn)

,provider_id,revenue
0,PVD-008,13774849.0
1,PVD-020,13379748.0
2,PVD-015,13251837.0
3,PVD-007,12239804.0
4,PVD-001,12202774.0
5,PVD-003,10342004.0
6,PVD-004,9991089.0
7,PVD-027,9829155.0
8,PVD-009,9577002.0
9,PVD-021,9162836.0


5.3 Margin Analysis

In [19]:
query = """
SELECT
provider_id,
ROUND(
SUM(actual_revenue_inr),
2
) AS revenue,
ROUND(
SUM(service_cost_inr),
2
) AS cost,
ROUND(
SUM(actual_revenue_inr)
-
SUM(service_cost_inr),
2
) AS margin
FROM treatment_outcomes
GROUP BY provider_id
ORDER BY margin DESC;
"""
pd.read_sql(query, conn)

,provider_id,revenue,cost,margin
0,PVD-008,13774849.0,7629183.0,6145666.0
1,PVD-015,13251837.0,7276119.0,5975718.0
2,PVD-020,13379748.0,7535729.0,5844019.0
3,PVD-001,12202774.0,6706491.0,5496283.0
4,PVD-007,12239804.0,6786951.0,5452853.0
5,PVD-003,10342004.0,5628749.0,4713255.0
6,PVD-004,9991089.0,5556161.0,4434928.0
7,PVD-027,9829155.0,5476367.0,4352788.0
8,PVD-009,9577002.0,5237173.0,4339829.0
9,PVD-022,8746069.0,4591608.0,4154461.0


# **Section 6: Country Benchmarking**
6.1 Country-wise Conversion

In [20]:
query = """
SELECT
country,
COUNT(*) AS patients,
SUM(treatment_completed) AS completed,
ROUND(
100.0 * SUM(treatment_completed)
/ COUNT(*),
2
) AS completion_rate
FROM patient_journey
GROUP BY country
ORDER BY completion_rate DESC;
"""
pd.read_sql(query, conn)

,country,patients,completed,completion_rate
0,Kenya,126,56,44.44
1,Malaysia,130,54,41.54
2,UAE,143,56,39.16
3,Oman,130,49,37.69
4,Bangladesh,136,51,37.50
5,Philippines,141,52,36.88
6,India,94,34,36.17
7,Sri Lanka,116,41,35.34
8,Nepal,133,47,35.34
9,Thailand,114,40,35.09


6.2 Country-wise Revenue

In [22]:
query = """
SELECT
    pj.country,
    ROUND(SUM(to_.actual_revenue_inr), 2) AS revenue
FROM
    patient_journey AS pj
JOIN
    treatment_outcomes AS to_
ON
    pj.patient_id = to_.patient_id
GROUP BY
    pj.country
ORDER BY
    revenue DESC;
"""
pd.read_sql(query, conn)

,country,revenue
0,Philippines,26747162.0
1,Malaysia,24068255.0
2,UAE,23152586.0
3,Bangladesh,22489480.0
4,Nepal,22316451.0
5,Kenya,21950483.0
6,Oman,21628726.0
7,Thailand,20285568.0
8,Sri Lanka,18580469.0
9,Tanzania,17297216.0


# **Section 7: Communication Analytics**
7.1 Channel Performance

In [23]:
query = """
SELECT
channel,
COUNT(*) AS interactions
FROM communication_logs
GROUP BY channel
ORDER BY interactions DESC;
"""
pd.read_sql(query, conn)

,channel,interactions
0,Web Form,1068
1,WhatsApp,1012
2,Referral Partner,1010
3,Phone,980
4,Chatbot,980
5,Email,975


7.2 Follow-up Status

In [24]:
query = """
SELECT
follow_up_status,
COUNT(*) AS total
FROM communication_logs
GROUP BY follow_up_status;
"""
pd.read_sql(query, conn)

,follow_up_status,total
0,Completed,2371
1,Not required,1198
2,Overdue,1241
3,Pending,1215


7.3 Average Response Delay

In [25]:
query = """
SELECT
channel,
ROUND(
AVG(response_delay_hours),
2
) AS avg_delay
FROM communication_logs
GROUP BY channel
ORDER BY avg_delay;
"""
pd.read_sql(query, conn)

,channel,avg_delay
0,Web Form,17.16
1,Chatbot,17.25
2,Referral Partner,17.65
3,WhatsApp,17.69
4,Phone,18.00
5,Email,18.50


# **Section 8: Executive SQL Dashboard**
8.1 Management KPI Query

In [26]:
query = """
SELECT
COUNT(*) AS total_patients,
SUM(treatment_completed) AS completed_treatments,
ROUND(
AVG(response_time_hours),
2
) AS avg_response_time
FROM patient_journey;
"""
pd.read_sql(query, conn)

,total_patients,completed_treatments,avg_response_time
0,1500,554,17.02


# **SQL Analytics Summary**

## Key SQL Reports

- Funnel KPI Dashboard
- Provider Performance Scorecard
- Revenue Analytics
- Margin Analysis
- Country Benchmarking
- Communication Metrics

## Business Value

- Identifies top-performing providers
- Tracks conversion funnel efficiency
- Measures revenue and profitability
- Benchmarks countries and markets
- Monitors communication effectiveness